# Walkthrough — L2: Bias-Variance Trade-Off & Model Selection

**O que este walkthrough faz:** explica célula por célula o notebook `L2_Bias_Variance_Model_Selection.ipynb`, com foco nas ferramentas Python/sklearn/Matplotlib usadas, por que cada escolha foi feita e como replicar os padrões.

**Conceitos matemáticos** já são explicados no notebook original — aqui o foco é no **código**.

---

## Roteiro
1. Setup — imports do sklearn e matplotlib
2. `make_pipeline` — encadeando transformações
3. `simular_decomposicao` — simulação de Monte Carlo com `axis=0`
4. `cross_val_score` — avaliação com K-Fold integrado
5. Visualizações de Ridge com grade de `lambda`
6. Comparação de modelos por tamanho de dados
7. O 4-Phase Workflow — implementação completa
8. `time.time` e benchmarking de modelos

---
## 1. Setup — Imports do sklearn e Matplotlib

### O que o código faz

```python
from sklearn.linear_model import Ridge, Lasso, LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import cross_val_score, KFold
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.dummy import DummyRegressor
from sklearn.datasets import make_regression
from sklearn.metrics import mean_squared_error
```

### Estrutura do sklearn

O sklearn organiza suas ferramentas em submódulos por função:

| Submódulo | Conteúdo |
|-----------|----------|
| `sklearn.linear_model` | Regressão linear, Ridge, Lasso, ElasticNet |
| `sklearn.preprocessing` | Transformações de features (PolynomialFeatures, StandardScaler) |
| `sklearn.pipeline` | Encadeamento de passos |
| `sklearn.model_selection` | CV, train_test_split, GridSearchCV |
| `sklearn.tree` | Decision Trees |
| `sklearn.ensemble` | Random Forest, Gradient Boosting |
| `sklearn.dummy` | Modelos de referência (baselines) |
| `sklearn.datasets` | Geradores de dados sintéticos |
| `sklearn.metrics` | Funções de avaliação |

### `warnings.filterwarnings('ignore')` — por que suprimir?

```python
import warnings
warnings.filterwarnings('ignore')
```

Alguns estimadores emitem `ConvergenceWarning` quando o solver não converge em `max_iter` iterações (comum com Lasso em dados mal escalados). Em produção, esses warnings são importantes. Em notebooks de exploração, podem poluir a saída. Use com critério.

---
## 2. `make_pipeline` — Encadeando Transformações

### O que o código faz

```python
modelo = make_pipeline(PolynomialFeatures(grau), LinearRegression())
modelo.fit(x_treino.reshape(-1, 1), y_treino)
y_pred = modelo.predict(x_plot.reshape(-1, 1))
```

### O que é um Pipeline sklearn

`make_pipeline(passo1, passo2, ...)` cria um estimador que aplica os passos em sequência:
1. `PolynomialFeatures(grau)` transforma `x` em `[1, x, x², ..., x^grau]`
2. `LinearRegression()` ajusta regressão linear nos features transformados

**A vantagem do Pipeline:** ao chamar `.fit(X, y)` no pipeline, ele chama automaticamente `.fit_transform` no transformador e `.fit` no estimador. Ao chamar `.predict(X)`, ele chama `.transform` e `.predict` em sequência.

**Por que usar Pipeline?**
- Evita **data leakage** em Cross-Validation: o `PolynomialFeatures` é ajustado só com os dados de treino de cada fold, não com os dados de validação
- Código mais limpo — um único objeto para treinar e prever

### `.reshape(-1, 1)` — convertendo 1D para 2D

```python
x_treino.reshape(-1, 1)
```

sklearn espera features com shape `(N, p)` — duas dimensões. `x_treino` tem shape `(N,)` — uma dimensão. `.reshape(-1, 1)` converte:
- `-1` diz ao NumPy para calcular essa dimensão automaticamente
- `1` especifica 1 coluna
- Resultado: shape `(N, 1)` — N amostras, 1 feature

### `mean_squared_error` — calculando MSE

```python
from sklearn.metrics import mean_squared_error
mse_treino = mean_squared_error(y_treino, y_pred_treino)
```

`mean_squared_error(y_true, y_pred)` calcula `(1/N) Σ (yᵢ - ŷᵢ)²`. Mais conveniente e legível que `np.mean((y - y_pred)**2)`.

---
## 3. `simular_decomposicao` — Monte Carlo com `axis=0`

### O que o código faz

```python
def simular_decomposicao(grau_poly, n_simulacoes=200, N=25, sigma_ruido=0.4):
    predicoes = np.zeros((n_simulacoes, len(x_teste)))  # shape: (200, 50)

    for i in range(n_simulacoes):
        # Gerar novo dataset aleatório
        x_tr = np.sort(np.random.uniform(0, 2, N))
        y_tr = f_true(x_tr) + np.random.normal(0, sigma_ruido, N)
        # Treinar modelo
        modelo = make_pipeline(PolynomialFeatures(grau_poly), Ridge(alpha=1e-6))
        modelo.fit(x_tr.reshape(-1, 1), y_tr)
        # Guardar previsões nesta simulação
        predicoes[i] = modelo.predict(x_teste.reshape(-1, 1))

    media_pred = predicoes.mean(axis=0)       # média sobre as 200 simulações
    bias2 = np.mean((f_verdadeiro_teste - media_pred) ** 2)
    variancia = np.mean(predicoes.var(axis=0))
    ...
```

### O array `predicoes` — 2D como coleção de experimentos

```python
predicoes = np.zeros((n_simulacoes, len(x_teste)))  # shape: (200, 50)
```

Cada linha é uma simulação, cada coluna é um ponto do eixo x. A linha `predicoes[i] = modelo.predict(...)` guarda o vetor de previsões da i-ésima simulação.

Após 200 simulações:
```
predicoes[0] = [ŷ(x₁) para dataset 0, ŷ(x₂) para dataset 0, ...]
predicoes[1] = [ŷ(x₁) para dataset 1, ŷ(x₂) para dataset 1, ...]
...
predicoes[199] = [ŷ(x₁) para dataset 199, ...]
```

### `axis=0` — operando ao longo das simulações

```python
media_pred = predicoes.mean(axis=0)  # shape: (50,)
variancia_por_ponto = predicoes.var(axis=0)  # shape: (50,)
```

`axis=0` opera ao longo da primeira dimensão (linhas = simulações). Para cada coluna (ponto x), calcula a média das 200 previsões. Resultado: um vetor de 50 médias.

- `axis=0` → colapsa as linhas (opera sobre as simulações)
- `axis=1` → colapsa as colunas (operaria sobre os pontos x)

### Calculando Bias² e Variance

```python
bias2 = np.mean((f_verdadeiro_teste - media_pred) ** 2)
variancia = np.mean(predicoes.var(axis=0))
```

- `Bias²`: distância entre a **média das previsões** e o valor verdadeiro, ao quadrado, depois média sobre todos os pontos x
- `Variance`: variância das previsões em cada ponto x, depois média sobre todos os pontos

`np.mean(predicoes.var(axis=0))` calcula a **variância esperada** — primeiro a variância pontual, depois a média espacial.

### Iteração com dicionário de resultados

```python
graus = [1, 3, 7, 14]
resultados = {g: simular_decomposicao(g) for g in graus}
```

Dict comprehension: para cada grau, roda a simulação e armazena. Acesso posterior: `resultados[7]` retorna a tuple `(bias2, variancia, irred, total, predicoes, x_teste, f_verdadeiro, media)` para grau 7.

---
## 4. `cross_val_score` — CV Integrado

### O que o código faz

```python
kf = KFold(n_splits=10, shuffle=True, random_state=42)

for g in graus_cv:
    modelo_cv = make_pipeline(PolynomialFeatures(g), Ridge(alpha=1e-4))
    scores = cross_val_score(modelo_cv, X_cv, y_cv, cv=kf,
                             scoring='neg_mean_squared_error')
    erros_cv_medio.append(-scores.mean())
    erros_cv_std.append(scores.std())
```

### `KFold` — definindo a estratégia de divisão

```python
kf = KFold(n_splits=10, shuffle=True, random_state=42)
```

- `n_splits=10` → 10 folds (treina em 9, valida em 1, repete 10 vezes)
- `shuffle=True` → embaralha os dados antes de dividir (importante quando os dados estão ordenados)
- `random_state=42` → reprodutibilidade do embaralhamento

**Por que `shuffle=True`?** Se os dados estiverem ordenados por alguma variável (ex: tempo), as primeiras linhas podem ser sistematicamente diferentes das últimas. Sem shuffle, um fold pode conter só exemplos "difíceis" ou "fáceis".

### `cross_val_score` — executa todo o CV

```python
scores = cross_val_score(modelo_cv, X_cv, y_cv, cv=kf,
                         scoring='neg_mean_squared_error')
```

Retorna um array de shape `(n_splits,)` — um score por fold.

**Por que `neg_mean_squared_error`?** sklearn usa a convenção de que **scores maiores são melhores**. Para o MSE, maior é pior — então sklearn o nega. `scores.mean()` dará um número negativo; `-scores.mean()` dá o MSE positivo.

**Internamente**, `cross_val_score` para cada fold:
1. Separa os dados em treino/validação
2. Chama `modelo.fit(X_treino, y_treino)` — treina um modelo **novo** a cada fold
3. Chama `modelo.predict(X_val)` e calcula o score
4. Descarta o modelo treinado naquele fold

**Vantagem do Pipeline aqui:** `make_pipeline(PolynomialFeatures, Ridge)` garante que o `PolynomialFeatures` seja ajustado apenas com os dados de treino de cada fold.

### Visualizando com banda de incerteza

```python
plt.fill_between(graus_cv,
                 erros_cv_medio - erros_cv_std,
                 erros_cv_medio + erros_cv_std,
                 alpha=0.2, color='tab:blue', label='±1 desvio padrão')
```

`fill_between(x, y_lower, y_upper)` preenche a área entre duas curvas. Mostra a **incerteza da estimativa** do CV — graus altos não só têm MSE maior mas também mais variabilidade entre folds.

---
## 5. Visualizações de Ridge — Grade de `lambda`

### O que o código faz

```python
lambdas = [0, 0.001, 0.01, 0.1, 1.0, 10.0]
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
axes = axes.ravel()

for ax, lam in zip(axes, lambdas):
    modelo_r = make_pipeline(PolynomialFeatures(grau_alto), Ridge(alpha=lam))
    modelo_r.fit(X_cv, y_cv)
    cv_s = cross_val_score(modelo_r, X_cv, y_cv, cv=5, scoring='neg_mean_squared_error')
    mse_tr = mean_squared_error(y_cv, modelo_r.predict(X_cv))
    ax.set_title(f'$\\lambda = {lam}$\\nMSE treino={mse_tr:.3f} | CV={-cv_s.mean():.3f}')
```

### `axes.ravel()` — achatando array 2D de eixos

```python
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
axes = axes.ravel()  # converte shape (2,3) para (6,)
```

`plt.subplots(2, 3)` retorna `axes` com shape `(2, 3)`. Para iterar com `zip(axes, lambdas)`, precisamos de um array 1D. `.ravel()` achata para `(6,)`.

Alternativa: `axes.flatten()` — funciona igual mas sempre cria uma cópia; `ravel()` cria uma view quando possível (mais eficiente).

### `\\n` em título de subplot

```python
ax.set_title(f'$\\lambda = {lam}$\\nMSE...')
```

- `\\n` dentro de f-string → produz `\n` → quebra de linha no título
- `$\\lambda$` → produz `$\lambda$` → renderiza como símbolo λ em LaTeX

### `np.logspace` para grades logarítmicas

```python
lambdas = np.logspace(-3, 4, 60)  # 60 valores de 10⁻³ a 10⁴
```

Gera valores espaçados uniformemente em **escala logarítmica**. `np.linspace(-3, 4, 60)` seria espaçamento linear — errado para λ, que opera em ordens de magnitude.

```
np.logspace(-3, 4, 60) → [0.001, 0.0015, ..., 1.0, ..., 10000]
np.linspace(-3, 4, 60) → [-3, -2.88, ..., 0, ..., 4]  ← valores negativos!
```

### `ax.semilogx` — eixo x em log

```python
plt.semilogx(lambdas_test, mse_treino_l2, ...)
```

Plota com eixo x em escala logarítmica. Equivalente a `ax.plot(...)` + `ax.set_xscale('log')`.

---
## 6. Comparação de Modelos por Tamanho de Dados

### O que o código faz

```python
def comparar_modelos_por_N(tamanhos_N, n_repeticoes=15):
    modelos = {
        'Ridge (L2)':        Ridge(alpha=1.0),
        'Random Forest':     RandomForestRegressor(n_estimators=50, random_state=7),
        'Gradient Boosting': GradientBoostingRegressor(n_estimators=50, random_state=7),
    }
    X_pool, y_pool = make_regression(n_samples=2000, n_features=10, noise=15.0,
                                      n_informative=5, random_state=42)
    for N in tamanhos_N:
        for nome, modelo in modelos.items():
            scores_rep = []
            for _ in range(n_repeticoes):
                idx = np.random.choice(len(X_pool), N, replace=False)
                X_s, y_s = X_pool[idx], y_pool[idx]
                cv_s = cross_val_score(modelo, X_s, y_s, cv=5,
                                       scoring='neg_mean_squared_error')
                scores_rep.append(-cv_s.mean())
            resultados_N[nome].append(np.mean(scores_rep))
```

### `make_regression` — dados sintéticos de regressão

```python
X_pool, y_pool = make_regression(
    n_samples=2000,       # número de amostras
    n_features=10,        # número total de features
    noise=15.0,           # desvio padrão do ruído
    n_informative=5,      # quantas features realmente importam
    random_state=42       # reprodutibilidade
)
```

`n_informative=5` cria 5 features verdadeiramente correlacionadas com y e 5 features de ruído puro. Isso simula datasets reais onde muitas features são irrelevantes.

### `np.random.choice` — amostragem sem reposição

```python
idx = np.random.choice(len(X_pool), N, replace=False)
X_s, y_s = X_pool[idx], y_pool[idx]
```

Seleciona `N` índices únicos aleatórios de `[0, 2000)`. `replace=False` → sem repetição (amostragem sem reposição). Usado para simular diferentes tamanhos de dataset a partir de um pool maior.

**Indexação fancy:** `X_pool[idx]` usa o array de índices para selecionar linhas — chamado de *fancy indexing* ou *advanced indexing* no NumPy.

### `modelos.items()` — iterando sobre dicionário

```python
for nome, modelo in modelos.items():
```

`.items()` retorna pares `(chave, valor)`. Permite iterar com nomes e objetos simultaneamente — padrão muito comum para comparar múltiplos estimadores.

### `DummyRegressor` — baseline ingênuo

```python
dummy = DummyRegressor(strategy='mean')
```

Prevê sempre a **média dos valores de treino** — independente de qualquer feature. É o "piso" mínimo: qualquer modelo que não supere o `DummyRegressor` é inútil.

Estratégias disponíveis:
- `'mean'` — prevê média
- `'median'` — prevê mediana  
- `'most_frequent'` — prevê classe mais frequente (classificação)
- `'constant'` — prevê valor fixo

---
## 7. O 4-Phase Workflow — Implementação Completa

### O que o código faz

```python
from sklearn.model_selection import train_test_split

X_wf, y_wf = make_regression(n_samples=2500, n_features=15, n_informative=8,
                              noise=20.0, random_state=42)

X_dev, X_test, y_dev, y_test = train_test_split(X_wf, y_wf, test_size=0.2, random_state=42)
```

### `train_test_split` — separação treino/teste

```python
X_dev, X_test, y_dev, y_test = train_test_split(
    X_wf, y_wf,
    test_size=0.2,     # 20% para teste, 80% para desenvolvimento
    random_state=42    # reprodutibilidade
)
```

Retorna 4 arrays. A nomenclatura `X_dev` (development set) = treino + validação — os dados com que você pode trabalhar livremente. `X_test` = tocado **uma única vez**, no final.

**Por que separar antes de qualquer análise?** Mesmo análise exploratória dos dados de teste pode introduzir viés inconsciente. O conjunto de teste deve ser uma caixa-preta até a avaliação final.

### Phase 3 — Baselines com print formatado

```python
scores_dummy = cross_val_score(dummy, X_dev, y_dev, cv=cv_wf,
                               scoring='neg_mean_squared_error')
mse_dummy = -scores_dummy.mean()

print(f"  Melhoria do Ridge vs. Dummy: {(1 - mse_ridge/mse_dummy)*100:.1f}% de redução do MSE")
```

**Melhoria relativa:** `(mse_base - mse_novo) / mse_base` mostra quanto do erro foi eliminado. Mais interpretável que comparar valores absolutos.

### Phase 4 — Candidate Shortlist com dicionário

```python
candidatos = {
    'Ridge (Baseline ML)':      Ridge(alpha=1.0),
    'Lasso (L1)':               Lasso(alpha=0.1, max_iter=5000),
    'Random Forest':            RandomForestRegressor(n_estimators=100, random_state=42),
    'Gradient Boosting':        GradientBoostingRegressor(n_estimators=100, random_state=42),
}

resultados_cand = {}
for nome, modelo in candidatos.items():
    s = cross_val_score(modelo, X_dev, y_dev, cv=cv_wf, scoring='neg_mean_squared_error')
    resultados_cand[nome] = {'mse': -s.mean(), 'std': s.std()}
```

O dicionário aninhado `resultados_cand[nome] = {'mse': ..., 'std': ...}` armazena múltiplas métricas por modelo. Acesso: `resultados_cand['Ridge (Baseline ML)']['mse']`.

**`sorted(resultados_cand.items(), key=lambda x: x[1]['mse'])`** — ordena os modelos por MSE usando uma função lambda como chave de ordenação. `x[1]` é o dicionário de métricas, `x[1]['mse']` é o valor a comparar.

### Avaliação final — usar o teste UMA vez

```python
modelo_final = GradientBoostingRegressor(n_estimators=100, random_state=42)
modelo_final.fit(X_dev, y_dev)   # treina em TODOS os dados de desenvolvimento
mse_teste = mean_squared_error(y_test, modelo_final.predict(X_test))
```

**Importante:** treina em `X_dev` (80% dos dados, não apenas 80% de 80%). O modelo final vê mais dados do que durante o CV — esperado. A avaliação é no `X_test` que nunca foi visto.

---
## 8. `time.time` — Benchmarking de Modelos

### O que o código faz

```python
import time

dados_exp = []
for nome, modelo in experimentos:
    t0 = time.time()
    modelo.fit(X_tr_m, y_tr_m)
    tempo_treino = time.time() - t0
    mse_m = mean_squared_error(y_te_m, modelo.predict(X_te_m))
    dados_exp.append({'nome': nome, 'mse': mse_m, 'tempo': tempo_treino})
```

### `time.time()` — cronômetro simples

`time.time()` retorna o tempo atual em segundos (float). A diferença entre duas chamadas dá o tempo decorrido. Para benchmarks mais precisos, use `time.perf_counter()` (mais alta resolução) ou `timeit` (para trechos muito rápidos).

### Lista de dicionários como acumulador

```python
dados_exp = []
dados_exp.append({'nome': nome, 'mse': mse_m, 'tempo': tempo_treino})
```

Padrão comum para acumular resultados heterogêneos: cada dict tem os mesmos campos, a lista tem um dict por experimento. Conversível para DataFrame: `pd.DataFrame(dados_exp)`.

### Subplots horizontais com `ax.barh`

```python
ax.barh(nomes_e, mses_e, color=cores_e, alpha=0.85)
ax.invert_xaxis()  # MSE menor = melhor → mostrar à esquerda
```

`barh` = barras horizontais. Útil quando os nomes dos modelos são longos — ficam no eixo y sem rotação. `invert_xaxis()` faz menor MSE aparecer à direita (visualmente "mais para frente").

---

## Resumo das Ferramentas

| Ferramenta | Uso principal no notebook |
|-----------|---------------------------|
| `make_pipeline(T, E)` | Encadeia transformação + estimador com CV seguro |
| `X.reshape(-1, 1)` | Converte array 1D para 2D (N,1) para sklearn |
| `cross_val_score(m, X, y, cv, scoring)` | CV completo em uma linha |
| `'neg_mean_squared_error'` | Scoring string para MSE (maior = melhor) |
| `KFold(n_splits, shuffle, random_state)` | Estratégia de CV reproduzível |
| `np.logspace(start, stop, num)` | Grid logarítmico para λ |
| `axes.ravel()` | Achata array 2D de eixos para 1D |
| `np.random.choice(n, N, replace=False)` | Amostragem sem reposição |
| `make_regression(n_samples, n_features, ...)` | Dataset sintético de regressão |
| `DummyRegressor(strategy='mean')` | Baseline ingênuo |
| `train_test_split(X, y, test_size, ...)` | Separação treino/teste |
| `time.time()` | Cronômetro wall-clock |
| `ax.barh(labels, values)` | Barras horizontais |
| `ax.invert_xaxis()` | Inverte eixo x |
| `fill_between(x, y_low, y_high)` | Banda de incerteza nos gráficos |

---

## Como Proceder ao Usar Estes Padrões

1. **Sempre separe o conjunto de teste primeiro** (`train_test_split`) antes de qualquer exploração.

2. **Use `cross_val_score` com `scoring='neg_mean_squared_error'`** e neque o resultado para obter o MSE positivo.

3. **Para comparar modelos:** use o mesmo `KFold` com `random_state` fixo para todos — garante os mesmos folds.

4. **Para simulações de Monte Carlo:** armazene todas as previsões em um array 2D `(n_sims, n_pontos)` e use `axis=0` para estatísticas sobre as simulações.

5. **Para dados de entrada 1D:** sempre `.reshape(-1, 1)` antes de passar para estimadores sklearn.